In [11]:
import anthropic
from anthropic.types import Message
from dotenv import load_dotenv

load_dotenv()

def add_user_messages(messages, message):
    messages.append(
        {
            "role": "user", 
            "content": message.content if isinstance(message, Message) else message
        }
    )

def add_assistant_messages(messages, message):
    messages.append(
        {
            "role": "assistant",
            "content": message.content if isinstance(message, Message) else message
        }
    )
    
def chat(messages, stop_sequences=[], temperature=1.0, system=None, tools=None):
    client = anthropic.Anthropic()
    parameters = {
        "model": "claude-haiku-4-5",
        "max_tokens": 512,
        "messages": messages,
        "temperature": temperature,
        "stop_sequences": stop_sequences
    }

    if tools:
        parameters["tools"] = tools

    if system:
        parameters["system"] = system

    response = client.messages.create(**parameters)
    return response


In [12]:
web_search_schema = {
    "type": "web_search_20260209",
    "name": "web_search", 
    "max_uses": 2,
    "allowed_callers": ["direct"] 
}

In [13]:
def run_tool(tool_request): 
    try:
        tool_output = ""
        if(tool_request.name == "get_current_datetime"):
            tool_output = get_current_datetime(**tool_request.input)
        if(tool_output == ""):
            raise ValueError("tool output is empty")
        return {
            "type": "tool_result",
            "tool_use_id": tool_request.id,
            "content": json.dumps(tool_output),
            "is_error": False
        }
    except Exception as e:
        return {
            "type": "tool_result",
            "tool_use_id": tool_request.id,
            "content": f"Error: {e}",
            "is_error": True
        }
def run_tools(message):
    print("Message content:", message.content)
    tool_requests = [
        block for block in message.content if block.type == "tool_use"
    ]

    tool_responses = []

    for tool_request in tool_requests:
        tool_responses.append(run_tool(tool_request))

    return tool_responses


In [14]:
def run_conversation(messages):
    while True:
        response = chat(messages, tools=[web_search_schema])
        add_assistant_messages(messages, response)

        print("Assistant response:", response)
        if response.stop_reason != "tool_use":
            print("Conversation stopped by model. Stop reason:", response.stop_reason)
            break

        tool_results = run_tools(response)
        add_user_messages(messages, tool_results)
        print("Tool results:", tool_results)
    return messages

In [ ]:
messages = []
add_user_messages(messages, "What is latest news about ClaudeCode?")
run_conversation(messages)
messages


 

Assistant response: Message(id='msg_014Xm387FmLfVEZ8u4uryC4d', container=None, content=[ServerToolUseBlock(id='srvtoolu_01CmG95somnP4m4C8Ggq2E8K', caller=None, input={'query': 'ClaudeCode latest news'}, name='web_search', type='server_tool_use'), WebSearchToolResultBlock(caller=DirectCaller(type='direct'), content=[WebSearchResultBlock(encrypted_content='EqgfCioIEBgCIiRlN2RkYzcwNS03YzU4LTRhOTMtYWE0NC04ODVkNDllYTE4ZWUSDEPASsrNcxz8ODLB+hoMHof4O+LH4VkT4bJZIjCs7TjxqwWYn0DVyMGXMQrXJ4mEzfYb05N5O3NV4VlQiULwS+SmNKthYzyIC03bos0qqx4y71QPEMWHnKFtra7j9glKz8pNZSGe49k+jcr0kRA9O4qEh0jMniz40zq+4S31wW4YlTminFG5C0ve6+avElcs7OvWgqSD0a1Rai0/Dof3cIEJa1zuoXzn5b9UlvFfnyCSR9GVQ33hmCWrv8mdd+bgOPU9FMbaS8YpiYBZMOZ7VX65G1xlHPomfE8rhZzqTt9w5mYl3FxAJ+slUzXVc/Xq0+gfndbu2Snf41IyD1IfDf5T7vomDtaX3R4EnAHT1El73+4yg5p2n3C10v3aNwAf0FRVl/KcQziyHM2kaC4pua8AxgCyF79KLJP6j3d7CDwrVtFFxBFvXf7FFY76v08XJez3EQJz7og0+OBVLiz1loPiR4wwVxHtA7Z12aqol9V1xaxeyNjzprvp5MIj1VI9o/G5Nen+17TZl+5bNwC9ZIbwxUeKFiW/WILvhZV7oXM5x5h5crnA3d6seSiVGFpDKXM

[{'role': 'user', 'content': 'What is latest news about ClaudeCode?'},
 {'role': 'assistant',
  'content': [ServerToolUseBlock(id='srvtoolu_01CmG95somnP4m4C8Ggq2E8K', caller=None, input={'query': 'ClaudeCode latest news'}, name='web_search', type='server_tool_use'),
   WebSearchToolResultBlock(caller=DirectCaller(type='direct'), content=[WebSearchResultBlock(encrypted_content='EqgfCioIEBgCIiRlN2RkYzcwNS03YzU4LTRhOTMtYWE0NC04ODVkNDllYTE4ZWUSDEPASsrNcxz8ODLB+hoMHof4O+LH4VkT4bJZIjCs7TjxqwWYn0DVyMGXMQrXJ4mEzfYb05N5O3NV4VlQiULwS+SmNKthYzyIC03bos0qqx4y71QPEMWHnKFtra7j9glKz8pNZSGe49k+jcr0kRA9O4qEh0jMniz40zq+4S31wW4YlTminFG5C0ve6+avElcs7OvWgqSD0a1Rai0/Dof3cIEJa1zuoXzn5b9UlvFfnyCSR9GVQ33hmCWrv8mdd+bgOPU9FMbaS8YpiYBZMOZ7VX65G1xlHPomfE8rhZzqTt9w5mYl3FxAJ+slUzXVc/Xq0+gfndbu2Snf41IyD1IfDf5T7vomDtaX3R4EnAHT1El73+4yg5p2n3C10v3aNwAf0FRVl/KcQziyHM2kaC4pua8AxgCyF79KLJP6j3d7CDwrVtFFxBFvXf7FFY76v08XJez3EQJz7og0+OBVLiz1loPiR4wwVxHtA7Z12aqol9V1xaxeyNjzprvp5MIj1VI9o/G5Nen+17TZl+5bNwC9ZIbwxUeKFiW/WILvhZV7oXM5